# 03 · Clasificador UNSPSC con transfer learning y búsqueda semántica (F4, F5, H2)

**ObservaCompras IA** · MVP · Aplicaciones de Inteligencia Artificial (MTI-493), UTFSM  
Profesor: Sr. Werner Creixell · Alumnos: Alejandra Cornejo Jara y Leoncio Cornejo Miranda

Aprendizaje **supervisado** multiclase de etiqueta única. Como entrenar una red de lenguaje desde cero exigiría millones de textos, se usa **Sentence-BERT multilingüe** (`paraphrase-multilingual-MiniLM-L12-v2`) como base congelada que extrae un vector de 384 dimensiones por licitación, de forma análoga a una base convolucional preentrenada. Sobre esos vectores se entrena Dense 128 (ReLU) → Dropout 0,3 → Dense n_clases (softmax), con Adam y `categorical_crossentropy`.

* Etiqueta: familia UNSPSC (4 dígitos) más frecuente entre los ítems de la licitación; se descartan familias con menos de 30 ejemplos.
* Validación 85/15 sobre 2023; prueba final con 2024 (una sola vez).
* Búsqueda: learning rate ∈ {1e-2, 1e-3, 1e-4} × batch size ∈ {64, 128, 256}; hasta 100 epochs.
* Línea base: TF-IDF + regresión logística.

> Se recomienda activar GPU en Colab (Entorno de ejecución → Cambiar tipo de entorno) para calcular los embeddings.

In [ ]:
# Preparación del entorno (Google Colab o local)
import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # El repositorio se clona en Drive para que datos y modelos persistan entre cuadernos
    RAIZ = "/content/drive/MyDrive/observacompras-ia"
    if not os.path.exists(RAIZ):
        subprocess.run(["git", "clone", "https://github.com/lcornejom/observacompras-ia", RAIZ], check=True)
    os.chdir(RAIZ)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(RAIZ)
sys.path.insert(0, os.path.join(RAIZ, "src"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Modo de periodos: "anio" (2023 y 2024 completos, como en el documento) o
# "demo" (solo un semestre: 1er trimestre = "2023", 2º trimestre = "2024")
MODO = "anio"

from observacompras import config as C
print("Raíz del proyecto:", RAIZ)

In [ ]:
import inspect
from observacompras import clasificador as K
print(inspect.getsource(K.construir_clasificador))
K.construir_clasificador(60).summary()

In [ ]:
from observacompras import pipeline_clf
import json
resultado = pipeline_clf.ejecutar(verbose=0)
print(json.dumps(resultado, indent=2, ensure_ascii=False))

## Curvas de pérdida y comparación con la línea base

In [ ]:
from IPython.display import Image, display
for f in ['clf_curva_perdida.png', 'clf_busqueda.png', 'clf_comparacion.png']:
    display(Image(str(C.DIR_FIGURAS / f)))

## F5 · Búsqueda semántica

Dos licitaciones son similares si sus vectores tienen alta similitud coseno.

In [ ]:
import numpy as np, pandas as pd
from observacompras.busqueda import BuscadorSemantico
from observacompras import embeddings as EMB
lic = pd.read_parquet(C.DIR_PROCESADOS / 'licitaciones_app.parquet')
E = np.load(C.DIR_PROCESADOS / 'embeddings_licitaciones.npy')
b = BuscadorSemantico(E, lic)
q = EMB.codificar(['mantención de equipos de rayos X'])[0]
display(b.por_vector(q, 10)[['similitud', 'NroLicitacion', 'Institucion', 'NombreLicitacion', 'familia']])

## Preparación de la vista de detalle de la aplicación

Precalcula la sugerencia del clasificador para las órdenes de compra de los 30 organismos más atípicos.

In [ ]:
from observacompras import pipeline_app
pipeline_app.ejecutar()